# Chapter 12 &mdash; The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$

**Concept 4 of the Chapter 12 decomposition:** *The Formal PDA $(Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$*

Seven components; $\Delta$ maps state &times; (input or $\varepsilon$) &times; stack symbol to a <i>set</i> of (state, push-string).

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Formal-PDA/Concept-Formal-PDA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$$P = (Q,\Sigma,\Gamma,\Delta,q_0,z_0,F)$$

* $Q$ &mdash; finite states;
* $\Sigma$ &mdash; input alphabet;
* $\Gamma$ &mdash; **stack** alphabet, a separate alphabet from $\Sigma$;
* $\Delta: Q\times\Sigma_\varepsilon\times\Gamma \to \mathcal{P}(Q\times\Gamma^*)$;
* $q_0$ &mdash; start state;
* $z_0 \in \Gamma$ &mdash; the **initial stack symbol**;
* $F \subseteq Q$ &mdash; final states.

Three details worth noticing. $\Delta$ returns a **set**, so the machine is
nondeterministic. It returns a **string** to push, so one move can push any bounded
number of symbols. And $\Gamma$ is **separate from** $\Sigma$ &mdash; the stack may use
symbols that never appear in the input, which is often the cleanest design.

## 2. Definitions

### A machine using its own stack alphabet

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
AnBn = md2mc('''PDA
!! Gamma = {#, A} -- 'A' never appears in the input.
I : a , # ; A#  -> I
I : a , A ; AA  -> I
I : b , A ; ''  -> M
M : b , A ; ''  -> M
M : '' , # ; #  -> F
I : '' , # ; #  -> F     !! the empty string
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### Reading the seven components off the dictionary

In [ ]:
def show_pda(P):
    for k in ['Q', 'Sigma', 'Gamma', 'q0', 'z0', 'F']:
        v = P[k]
        print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))
    print("Delta :")
    for (q, i, s), outs in sorted(P["Delta"].items()):
        print("   (%s, %r, %s) -> %s" % (q, i, s, sorted(outs)))

## 3. Tests

All seven components.

In [ ]:
show_pda(AnBn)
assert AnBn["z0"] in AnBn["Gamma"]

$\Gamma$ is **separate** from $\Sigma$: the stack symbol `A` is not an input symbol.

In [ ]:
print("Sigma :", sorted(AnBn["Sigma"]))
print("Gamma :", sorted(AnBn["Gamma"]))
print("Gamma - Sigma :", sorted(AnBn["Gamma"] - AnBn["Sigma"]))
assert 'A' in AnBn["Gamma"] and 'A' not in AnBn["Sigma"]

$\Delta$ returns a **set** of (state, push-string) pairs.

In [ ]:
for k, outs in sorted(AnBn["Delta"].items()):
    assert isinstance(outs, set)
    for (q2, push) in outs:
        assert isinstance(push, str)
print("every Delta value is a set of (state, push-string) pairs")

And the machine works.

In [ ]:
def in_anbn(s):
    k = len(s) - len(s.lstrip('a'))
    return s == 'a'*k + 'b'*(len(s)-k) and k == len(s)-k
from itertools import product
strs = [''.join(p) for k in range(7) for p in product('ab', repeat=k)]
bad = [s for s in strs if pda_accepts(AnBn, s, STKMAX=9) != in_anbn(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

The push-string lets one move push several symbols.

In [ ]:
Two = md2mc('''PDA
I : a , # ; XY#  -> I
I : b , X ; ''   -> I
I : b , Y ; ''   -> I
I : '' , # ; #   -> F
''')
print("accepts 'abb' ?", pda_accepts(Two, 'abb', STKMAX=6))
assert pda_accepts(Two, 'abb', STKMAX=6)
print("one 'a' pushed TWO symbols, so two 'b's are needed to clear them.")

## 4. Exercises


1. Why is $z_0$ part of the definition rather than just "start with an empty stack"?
2. What is the largest push-string in your own PDA designs? Does it matter?
3. Write the DFA five-tuple as a PDA seven-tuple. What do you set $\Gamma$ to?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter12/Concept-Formal-PDA')